In [1]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "04_embeddings_models"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [2]:
# 2. Carregar dados processados
import pandas as pd, numpy as np

df_ibov = pd.read_csv(os.path.join(PROC_PATH, "ibovespa_clean.csv"))
df_news = pd.read_csv(os.path.join(PROC_PATH, "noticias_clean.csv"))

df_ibov["data"] = pd.to_datetime(df_ibov["data"])
df_news["data"] = pd.to_datetime(df_news["data"])

print("IBOV:", df_ibov.shape, "NEWS:", df_news.shape)
display(df_news.head())

IBOV: (20, 2) NEWS: (10, 6)


,data,fonte,titulo,texto,sentimento,clean_text
0,2024-01-02,exemplo,Titulo 1,Ibovespa sobe com otimismo no mercado internac...,1,ibovespa sobe otimismo mercado internacional
1,2024-01-03,exemplo,Titulo 2,Bolsa cai após dados fracos da economia chinesa.,0,bolsa cai após dados fracos economia chinesa
2,2024-01-04,exemplo,Titulo 3,Petróleo em alta puxa ações da Petrobras para ...,1,petróleo alta puxa ações petrobras cima
3,2024-01-05,exemplo,Titulo 4,Queda do dólar beneficia empresas exportadoras.,1,queda dólar beneficia empresas exportadoras
4,2024-01-08,exemplo,Titulo 5,Setor bancário recua com aversão ao risco.,0,setor bancário recua aversão risco


In [3]:
# 3. Instalar & carregar modelo de embeddings (FinBERT ou BERTimbau)
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

# modelo multilÃ­ngue (pt-en) robusto
model = SentenceTransformer("all-MiniLM-L6-v2")

# gerar embeddings para as notÃ­cias
embeddings = model.encode(df_news["clean_text"].fillna("").tolist(), show_progress_bar=True)

import pandas as pd
emb_df = pd.DataFrame(embeddings)
emb_df["data"] = df_news["data"].values

# Agregar por dia (mÃ©dia dos embeddings das notÃ­cias do dia)
daily_emb = emb_df.groupby("data").mean().reset_index()
print("Daily Embeddings shape:", daily_emb.shape)

DEPRECATION: Loading egg at c:\python311\lib\site-packages\vboxapi-1.0-py3.11.egg is deprecated. pip 23.3 will enforce this behaviour change. A possible replacement is to use pip for package installation..

[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ander\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Daily Embeddings shape: (10, 385)


In [4]:
# 4. Merge com IBOV e criar target
df = pd.merge(df_ibov.sort_values("data"), daily_emb, on="data", how="left").fillna(0)
df["ret1"] = df["close"].pct_change().shift(-1)           # retorno do dia seguinte
df = df.dropna().copy()
df["y"] = (df["ret1"] > 0).astype(int)

X = df.drop(columns=["data","close","ret1","y"]).values
y = df["y"].values

print("Final dataset:", X.shape, y.shape)

Final dataset:

 (19, 384) (19,)


In [5]:
# 5. Modelos: Logistic, RandomForest, XGBoost
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

models = {
    "Logistic": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                             subsample=0.8, colsample_bytree=0.8, random_state=42)
}

tscv = TimeSeriesSplit(n_splits=3)
results = {}

for name, clf in models.items():
    aucs, mdas = [], []
    for tr, te in tscv.split(X):
        clf.fit(X[tr], y[tr])
        proba = clf.predict_proba(X[te])[:,1]
        pred  = (proba > 0.5).astype(int)
        aucs.append(roc_auc_score(y[te], proba))
        mdas.append(accuracy_score(y[te], pred))
    results[name] = {"AUC": np.mean(aucs), "MDA": np.mean(mdas)}

print("Resultados mÃ©dios (Embeddings):")
print(results)

Resultados mÃ©dios (Embeddings):
{'Logistic': {'AUC': np.float64(0.5555555555555556), 'MDA': np.float64(0.5833333333333334)}, 'RandomForest': {'AUC': np.float64(0.4444444444444445), 'MDA': np.float64(0.6666666666666666)}, 'XGBoost': {'AUC': np.float64(0.5), 'MDA': np.float64(0.5833333333333334)}}


In [6]:
# 6. Salvar resultados em JSON (para o dashboard)
import json, os

nb_name = "04_embeddings_models"

# results jÃ¡ estÃ¡ definido pelos modelos
results_dict = {k: {"AUC": float(v["AUC"]), "MDA": float(v["MDA"])} for k, v in results.items()}

out_file = os.path.join(PROC_PATH, f"results_{nb_name}.json")
with open(out_file, "w") as f:
    json.dump(results_dict, f, indent=4)

print(f"âœ… Resultados salvos em {out_file}")

âœ… Resultados salvos em C:\Users\ander\OneDrive\TCC_USP\data_processed\results_04_embeddings_models.json
